In [15]:
# ==============================
# 1. INSTALL
# ==============================
# pip install sentence-transformers rapidfuzz

# ==============================
# 2. IMPORTS
# ==============================
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
import re



In [16]:
# ==============================
# 3. LOAD MODEL
# ==============================
model = SentenceTransformer('all-MiniLM-L6-v2')

# ==============================
# 4. NORMALIZATION
# ==============================
def normalize_text(text):
    text = text.lower()

    replacements = {
        "0": "o", "1": "i", "3": "e",
        "@": "a", "$": "s"
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6995.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
#==============================
# 5. GROOMING PATTERNS (CORE)
# ==============================
GROOMING_PATTERNS = [
    "let's keep this between us",
    "don't tell your parents",
    "this is our secret",
    "you are special to me",
    "i understand you better than others",
    "we should meet privately",
    "just the two of us",
    "don't tell anyone",
    "keep this secret",
    "come alone"
]

pattern_embeddings = model.encode(GROOMING_PATTERNS)



In [18]:
# ==============================
# 6. SEMANTIC SCORE
# ==============================
def semantic_score(messages):
    text = " ".join(messages[-5:])
    text = normalize_text(text)

    emb = model.encode(text)
    scores = util.cos_sim(emb, pattern_embeddings)

    return float(scores.max())



In [ ]:
# ==============================
# 7. RULE BOOST
# ==============================
def rule_score(messages):
    score = 0

    for msg in messages:
        msg = normalize_text(msg)   # 🔥 IMPORTANT FIX

        if "secret" in msg or "dont tell" in msg:
            score += 2

        if "meet" in msg and "alone" in msg:
            score += 2

        if "private" in msg:
            score += 1.5

        if "special" in msg or "trust me" in msg:
            score += 1

    return min(score / 5, 1)



In [ ]:
def critical_override(messages):
    text = normalize_text(" ".join(messages))

    if "dont tell" in text and ("meet" in text or "alone" in text):
        return True

    if "secret" in text and "private" in text:
        return True

    return False

In [20]:
# ==============================
# 8. FUZZY (LIGHT)
# ==============================
def fuzzy_score(text):
    phrases = ["dont tell", "secret", "meet alone"]
    score = 0

    for p in phrases:
        score = max(score, fuzz.partial_ratio(text, p) / 100)

    return score * 0.5  # reduce impact



In [21]:
# ==============================
# 9. CONTEXT SAFETY FILTER
# ==============================
SAFE_WORDS = [
    "recipe", "project", "meeting",
    "team", "work", "assignment",
    "friends", "group"
]

def context_penalty(text):
    for word in SAFE_WORDS:
        if word in text:
            return 0.3
    return 0



In [ ]:
# ==============================
# 10. FINAL DETECTION
# ==============================
def detect_grooming(messages):
    text = normalize_text(" ".join(messages))

    sem = semantic_score(messages)
    rule = rule_score(messages)
    fuzzy = fuzzy_score(text)
    penalty = context_penalty(text)

    final_score = (
        0.6 * sem +
        0.3 * rule +
        0.1 * fuzzy
    )

    final_score = max(0, final_score - penalty)

    # 🔥 override logic
    if critical_override(messages):
        final_score = max(final_score, 0.85)

    if final_score > 0.7:
        level = "HIGH"
    elif final_score > 0.4:
        level = "MEDIUM"
    else:
        level = "LOW"

    return {
        "risk_score": round(final_score, 3),
        "level": level,
        "signals": {
            "semantic": sem,
            "rule": rule,
            "fuzzy": fuzzy
        }
    }



In [25]:
# ==============================
# 11. TEST
# ==============================
if __name__ == "__main__":
    tests = [
        ["hey how was your day?", "see you tomorrow"],
        ["you are special to me", "dont tell anyone"],
        ["this is a secret recipe", "lets meet at a cafe alone"],
        ["d0nt t3ll any0ne", "lets m33t al0ne"]
    ]

    for t in tests:
        print(t)
        print(detect_grooming(t))
        print("-" * 50)

['hey how was your day?', 'see you tomorrow']
{'risk_score': 0.226, 'level': 'LOW', 'signals': {'semantic': 0.3356887996196747, 'rule': 0.0, 'fuzzy': 0.25}}
--------------------------------------------------
['you are special to me', 'dont tell anyone']
{'risk_score': 0.72, 'level': 'HIGH', 'signals': {'semantic': 0.8171474933624268, 'rule': 0.6, 'fuzzy': 0.5}}
--------------------------------------------------
['this is a secret recipe', 'lets meet at a cafe alone']
{'risk_score': 0.248, 'level': 'LOW', 'signals': {'semantic': 0.4308024048805237, 'rule': 0.8, 'fuzzy': 0.5}}
--------------------------------------------------
['d0nt t3ll any0ne', 'lets m33t al0ne']
{'risk_score': 0.444, 'level': 'MEDIUM', 'signals': {'semantic': 0.6566572189331055, 'rule': 0.0, 'fuzzy': 0.5}}
--------------------------------------------------
